# Simulação de um sistema de elevação de petróleo com ESP\*

\*bomba elétrica submersível, do inglês, Electric Submersible Pump.


## Configurações iniciais


In [1]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
import lib.model as m

import matplotlib as mpl

mpl.rcParams["lines.linewidth"] = 2


# --- Condições iniciais ---
Vv0 = 0.03322057460555981  # vazão volumétrica no tubo vertical [m³/s]
pm0 = 2910127.158128174  # Pressão no manifold [Pa]
Vt0 = 0.04316711190345594  # vazão volumétrica no tubo horizontal [m³/s]
fp0 = 60.0  # Frequência de rotação da ESP [Hz]
y0 = [Vv0, pm0, Vt0, fp0]

t_span = (0, 30)
t = np.linspace(*t_span, 10000)


## Definição do modelo sem controlador


In [2]:
def manual_fp(t):
    return np.where(t > 1, 81.57732024513601, 60.0)


def model(t, y):
    Vv, pm, Vt = y
    fp = np.where(t > 1, 81.57732024513601, 60.0)

    dVv_dt, dpm_dt, dVt_dt = m.EDOs(t, [Vv, pm, Vt], fp)

    return [dVv_dt, dpm_dt, dVt_dt]


## Definição do modelo com controlador


In [3]:
def pm_SP(t):
    """Setpoint para o controlador"""
    SP = np.full_like(t, pm0)  # Valor padrão (t <= 30)
    SP = np.where(t > 1, 35e5, SP)
    # SP = np.where(t > 10, 27e5, SP)
    return SP


def model_PI(t, y):
    Vv, pm, Vt, fp = y

    dVv_dt, dpm_dt, dVt_dt = m.EDOs(t, [Vv, pm, Vt], fp)

    # controle PI
    erro = pm_SP(t) - pm
    Kp = 0.00015
    Ki = 0.00040
    dfp_dt = -Kp * dpm_dt + Ki * erro

    return [dVv_dt, dpm_dt, dVt_dt, dfp_dt]


## Simulação e plotagem dos dados


In [4]:
sol = solve_ivp(model, t_span, y0[:3], t_eval=t, method="LSODA")
Vv, pm, Vt = sol.y

print("Vv:", Vv[-1])
print("pm:", pm[-1])
print("Vt:", Vt[-1])

sol_PI = solve_ivp(model_PI, t_span, y0, t_eval=t, method="LSODA")
Vv_PI, pm_PI, Vt_PI, fp_PI = sol_PI.y


# --- Plotagem das saídas do sistema---
plt.figure(figsize=(12, 6))
plt.subplot(2, 1, 1)
plt.plot(t, Vv, label="Vazão no tubo horizontal ($\\dot{V}_v$) sem controlador")
plt.plot(t, Vt, label="Vazão no tubo vertical ($\\dot{V}_t$) sem controlador")

plt.plot(t, Vv_PI, label="Vazão no tubo horizontal ($\\dot{V}_v$) com controlador")
plt.plot(t, Vt_PI, label="Vazão no tubo vertical ($\\dot{V}_t$) com controlador")
plt.ylabel("Vazão / m$^3\\cdot$s$^{-1}$")
plt.legend()

plt.subplot(2, 1, 2)
plt.plot(t, pm, label="Pressão no Manifold ($p_m$) sem controlador")
plt.plot(t, pm_PI, label="Pressão no Manifold ($p_m$) com controlador")
plt.ylabel("Pressão / Pa")
plt.xlabel("Tempo / s")
plt.legend()
plt.tight_layout()
plt.savefig("../figures/simulation.png")
plt.close()

# --- Gráficos com variável controlada e variável manipulada ---
_, axs = plt.subplots(2, figsize=(12, 5), layout="constrained")

axs[0].plot(t, pm / 1e5, label="$p_m$ sem controlador")
axs[0].plot(t, pm_PI / 1e5, label="$p_m$ com controlador")
axs[0].plot(t, pm_SP(t) / 1e5, "--", label="Setpoint", color="red")
axs[0].set_xlabel("Tempo / s")
axs[0].set_ylabel("Pressão no manifold / bar")
axs[0].legend()
axs[0].grid()

axs[1].plot(t, manual_fp(t), label="$f_p$(t) sem controlador")
axs[1].plot(t, fp_PI, label="$f_p$(t) com controlador")
axs[1].set_xlabel("Tempo / s ")
axs[1].set_ylabel("Frequência da ESP / Hz")
axs[1].legend()
axs[1].grid()

plt.savefig("../figures/comparison.png")
plt.close()


Vv: 0.04269372545295728
pm: 3516052.7591688046
Vt: 0.05543583166454764


## Medindo tempo de estabilização


In [5]:
from lib.utils import tempo_de_estabilização

tempo_sem_controle = tempo_de_estabilização(t, pm, pm_SP(t)) - 1
print(f"Tempo de estabilização sem controle: {tempo_sem_controle:.2f} s")

tempo_com_controle = tempo_de_estabilização(t, pm_PI, pm_SP(t)) - 1
print(f"Tempo de estabilização com controle: {tempo_com_controle:.2f} s")


Tempo de estabilização sem controle: 23.00 s
Tempo de estabilização com controle: 1.01 s


In [9]:
from lib.utils import calcular_metricas_erro_normalizado

SP_t = np.array([pm_SP(ti) for ti in sol.t])
iae_n, ise_n, itae_n = calcular_metricas_erro_normalizado(sol.t, sol.y[1, :], SP_t)
print(f"IAE Normalizado  = {iae_n:.4f}")
print(f"ISE Normalizado  = {ise_n:.4f}")
print(f"ITAE Normalizado = {itae_n:.4f}")

iae_n_PI, ise_n_PI, itae_n_PI = calcular_metricas_erro_normalizado(sol_PI.t, sol_PI.y[1, :], SP_t)
print(f"IAE Normalizado com PI  = {iae_n_PI:.4f}")
print(f"ISE Normalizado  com Pi= {ise_n_PI:.4f}")
print(f"ITAE Normalizado com PI= {itae_n_PI:.4f}")

IAE Normalizado  = 2.0226
ISE Normalizado  = 0.4597
ITAE Normalizado = 13.2864
IAE Normalizado com PI  = 0.0782
ISE Normalizado  com Pi= 0.0067
ITAE Normalizado com PI= 0.1663
